<a href="https://colab.research.google.com/github/JonnaBauer/DeepLearningProject/blob/main/ModernBERT_FRB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1 - Enable GPU

# Step 2 - Install Dependencies

In [1]:
# !pip install -q transformers peft datasets==2.21.0 evaluate accelerate scikit-learn
!pip install -q transformers datasets==2.21.0 evaluate accelerate scikit-learn -U peft torchao

In [ ]:
import time, numpy as np, torch
from datasets import load_dataset
from transformers import (AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate

import torch
import torchvision
import datasets

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("datasets:", datasets.__version__)

print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

# Step 3 - Load and Tokenize SST-2

In [ ]:
dataset = load_dataset("stanfordnlp/sst2")
MODEL_NAME = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True,
                     padding="max_length", max_length=128)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch",
    columns=["input_ids", "attention_mask", "labels"])
print("Dataset ready")

# Step 4 - Metric

In [4]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

# Step 5 - Experiment Config

In [5]:
# ── change only these four values ──────────────────────────
USE_LORA      = True   # True = LoRA  |  False = full fine-tuning
LORA_RANK     = 16       # 4, 8, or 16  (ignored if USE_LORA=False)
NUM_EPOCHS    = 3       # 1, 2, or 3
TRAIN_SAMPLES = None    # None = full  |  500, 1000, 5000
# ───────────────────────────────────────────────────────────

# Step 6 - Load Model and Apply LoRA

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2)

if USE_LORA:
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_RANK,
        lora_alpha=LORA_RANK * 2,
        target_modules=["Wqkv", "dense"],
        lora_dropout=0.1,
        bias="none"
    )
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
else:
    n = sum(p.numel() for p in model.parameters())
    print(f"Full fine-tuning — trainable params: {n:,}")

# Step 7 - Subsample

In [7]:
train_ds = tokenized["train"]
eval_ds  = tokenized["validation"]

if TRAIN_SAMPLES:
    train_ds = train_ds.shuffle(seed=42).select(range(TRAIN_SAMPLES))
    print(f"Using {TRAIN_SAMPLES} training samples")
else:
    print(f"Using full training set ({len(train_ds)} samples)")

Using full training set (67349 samples)


# Step 8 - Train

In [ ]:
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    seed=42,
    report_to="none"
)

trainer = Trainer(model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    compute_metrics=compute_metrics)

start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"Training time: {elapsed/60:.1f} min")

# Step 9 - Results

In [ ]:
results = trainer.evaluate()
trainable = sum(p.numel() for p in model.parameters()
                if p.requires_grad)

print("=" * 40)
print(f"USE_LORA:         {USE_LORA}")
print(f"LORA_RANK:        {LORA_RANK if USE_LORA else 'N/A'}")
print(f"NUM_EPOCHS:       {NUM_EPOCHS}")
print(f"TRAIN_SAMPLES:    {TRAIN_SAMPLES or 'full'}")
print(f"Val accuracy:     {results['eval_accuracy']:.4f}")
print(f"Trainable params: {trainable:,}")
print(f"Training time:    {elapsed/60:.1f} min")
print("=" * 40)